In [1]:
import os, json, glob
import numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

@keras.utils.register_keras_serializable(package="oral")
class ImageNetPreprocess(layers.Layer):
    def __init__(self, mode="none", **kwargs):
        super().__init__(**kwargs); self.mode = mode
    def call(self, x):
        if self.mode == "none": return x
        return keras.applications.imagenet_utils.preprocess_input(x, mode=self.mode)
    def get_config(self):
        cfg = super().get_config(); cfg.update(mode=self.mode); return cfg

MODEL = glob.glob("/kaggle/input/**/ResNet50_tuned_best.keras", recursive=True)[0]
VALJ  = glob.glob("/kaggle/input/**/ResNet50_tuned_val.json", recursive=True)
DATA  = os.path.dirname(glob.glob("/kaggle/input/**/manifest.csv", recursive=True)[0])
print("model:", MODEL, "|", round(os.path.getsize(MODEL)/1e6, 1), "MB")
print("val json:", VALJ)
if VALJ: print({k: v for k, v in json.load(open(VALJ[0])).items() if k in ("val_accuracy", "val_macro_f1", "config")})
m = keras.models.load_model(MODEL, compile=False)
print("input:", m.input_shape, "| output:", m.output_shape)

model: /kaggle/input/notebooks/mazenayad/step4-tune-resnet50/models/ResNet50_tuned_best.keras | 95.1 MB
val json: ['/kaggle/input/notebooks/mazenayad/step4-tune-resnet50/results/ResNet50_tuned_val.json']
{'val_accuracy': 0.9437751004016064, 'val_macro_f1': 0.8957046093726125, 'config': {'lr_ft': 0.0001, 'unfreeze_last': 80, 'dropout': 0.5, 'batch': 16}}


I0000 00:00:1792928220.268033      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1792928220.271779      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


input: (None, 224, 224, 3) | output: (None, 6)


In [2]:
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score
CLASS_NAMES = json.load(open(os.path.join(DATA, "meta.json")))["class_names"]
H, W = m.input_shape[1:3]
def predict(img):
    x = tf.image.resize(tf.convert_to_tensor(np.asarray(img.convert("RGB")), tf.float32), (H, W))
    return m.predict(x[None], verbose=0)[0]

y, p = [], []
for ci, c in enumerate(CLASS_NAMES):
    for f in sorted(os.listdir(f"{DATA}/test/{c}")):
        y.append(ci); p.append(predict(Image.open(f"{DATA}/test/{c}/{f}")).argmax())
print("images:", len(y), "| acc", round(accuracy_score(y, p), 3), "| macro-F1", round(f1_score(y, p, average="macro"), 3))

images: 506 | acc 0.947 | macro-F1 0.884
